# Part 2 — Embeddings

*Turning meaning into numbers so a computer can search by meaning, not by spelling.*

📖 Read the essay: https://www.mefby.com/essays/embeddings  
🐍 Companion script: `embeddings.py`

**What you'll build**

1. `tokenize` + a tiny **vocabulary** over our running example.
2. **One-hot encoding** — and a first-hand look at why it captures *identity*, not meaning.
3. **Bag-of-words** — a step up that still only sees shared *words*, not shared *meaning*.
4. A **guarded dense embedder**: the real `sentence-transformers` model when it's available, a transparent deterministic stand-in otherwise.
5. The chunk-vs-query comparison that finally *works*, plus the classic `king − man + woman ≈ queen` arithmetic.

> This notebook runs **fully offline**. It needs only `numpy`. If a real embedding model is installed and cached it is used automatically; otherwise every cell still runs against a transparent, dependency-free fallback. You never need a network connection or an API key.

Coming from **Part 1 — Why RAG Exists**, you already have the seven-station pipeline in your head: *document, chunk, embed, store, retrieve, augment, generate*. Today we slow down on the third station, **embed**, because everything after it depends on getting this one right.

## Setup

Just two imports from the standard library plus `numpy`. We pull in `hashlib` and `re` now because the offline fallback later hashes tokens into vector dimensions, and `re` does our word-splitting. Nothing here touches the network.

In [ ]:
import hashlib
import re

import numpy as np

# The running example from the essay: one chunk of a refund policy, and a user
# question phrased in completely different words. Keyword search struggles here;
# a good embedding should still place them close together in meaning-space.
CHUNK = "Refunds are accepted within 30 days of purchase."
QUERY = "What is our refund window?"

print("chunk:", CHUNK)
print("query:", QUERY)

## The question underneath the whole pipeline

You and I read those two lines and instantly see they're about the same thing. But look at them as a computer does, as raw characters: they share almost no words. The question never says *30*, never says *days*, never even says *accepted*. The chunk never says *window*.

So here's the problem this whole part exists to solve: **the user asks in their words, your documents are written in someone else's words, and "relevant" has to mean *similar in meaning*, not *similar in spelling*.** A computer can't compare meanings directly — it only compares numbers, blazingly fast. So the entire game is: turn each piece of text into numbers in a way that *preserves meaning*, then do arithmetic on the numbers instead of the words. Let's watch the two naive ways of doing that fail first, because understanding *why* they fail is the cleanest motivation for embeddings.

## Step 0 — tokens and a vocabulary

Before any encoding we need two helpers. `tokenize` lowercases the text and pulls out plain words (the essay stays at the level of words, so we ignore punctuation). A **vocabulary** is the list of every distinct word the system might see — we sort it so each word lands in a *stable* slot, which makes every vector below reproducible.

In [ ]:
def tokenize(text):
    # Lowercase words/numbers only; the essay stays at the level of plain words.
    return re.findall(r"[a-z0-9]+", text.lower())


def build_vocabulary(corpus):
    # Stable, sorted vocabulary so slot positions are reproducible.
    vocab = sorted({word for doc in corpus for word in tokenize(doc)})
    return {word: i for i, word in enumerate(vocab)}


corpus = [CHUNK, QUERY]
vocab = build_vocabulary(corpus)
print("tokens(chunk):", tokenize(CHUNK))
print("tokens(query):", tokenize(QUERY))
print(f"vocabulary size for our two tiny sentences: {len(vocab)} words")

## Step 1 — Naive idea #1: one-hot encoding

Represent a single word as a list with one slot *per vocabulary word*, all zeros except a single `1` in *this* word's slot. That's **one-hot encoding** — one slot is "hot" (set to 1) and the rest are cold. For real text the vocabulary is tens or hundreds of thousands of words, so the list is enormous and almost entirely empty: a **sparse vector** (a vector whose entries are mostly zero). A *vector* is just an ordered list of numbers.

In [ ]:
def one_hot(word, vocab):
    # All zeros except a single 1 at this word's position -> a SPARSE vector.
    vec = np.zeros(len(vocab), dtype=int)
    if word in vocab:
        vec[vocab[word]] = 1
    return vec


oh_refunds = one_hot("refunds", vocab)
print("one_hot('refunds') =", oh_refunds.tolist())
print(f"  {int(oh_refunds.sum())} hot slot out of {len(oh_refunds)}"
      "  -> sparse: all identity, no meaning")

Here is the fatal flaw, made literal. Every word sits in its own private slot, touching nothing else. So by *any* measure of closeness, `king` is exactly as far from `queen` as it is from `banana`. The encoding captured *identity* — the bare fact that this word is not that word — and never captured meaning. Let's prove it on a toy vocabulary.

In [ ]:
royalty_vocab = build_vocabulary(["king queen banana"])
v_king   = one_hot("king",   royalty_vocab)
v_queen  = one_hot("queen",  royalty_vocab)
v_banana = one_hot("banana", royalty_vocab)

# Overlap between two one-hot vectors = their dot product (1 if same word, else 0).
print("king . queen  =", int(v_king @ v_queen))
print("king . banana =", int(v_king @ v_banana))
print("-> both 0: king is no closer to queen than to banana. Identity, not meaning.")

## Step 2 — Naive idea #2: bag-of-words

A step up: encode a *whole passage* by **counting** how many times each vocabulary word appears in it. We call it a "bag" because we throw all the words into a sack and count them, keeping no record of order. It's the ancestor of classic keyword search — genuinely useful — but for *meaning* it hits the same wall: two passages look similar only if they reuse the *same words*.

In [ ]:
def bag_of_words(text, vocab):
    vec = np.zeros(len(vocab), dtype=int)
    for word in tokenize(text):
        if word in vocab:
            vec[vocab[word]] += 1
    return vec


bow_chunk = bag_of_words(CHUNK, vocab)
bow_query = bag_of_words(QUERY, vocab)
print(f"bag-of-words(chunk) nonzero slots: {int((bow_chunk > 0).sum())}")
print(f"bag-of-words(query) nonzero slots: {int((bow_query > 0).sum())}")

To compare two vectors we need one number for "how aligned are they?". We'll use **cosine similarity**: `1.0` means same direction, `0.0` unrelated, `-1.0` opposite. (Cosine is the whole subject of Part 3 — here we just borrow it to score the comparisons.) We also add a tiny `unit` helper that scales a vector to length 1, used later.

In [ ]:
def unit(vec):
    norm = np.linalg.norm(vec)
    return vec if norm == 0 else vec / norm


def cosine(a, b):
    # 1.0 == same direction, 0.0 == unrelated, -1.0 == opposite.
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return 0.0 if denom == 0 else float(np.dot(a, b) / denom)


print(f"cosine(chunk, query) under bag-of-words = {cosine(bow_chunk, bow_query):.3f}")
print("-> near zero: almost no shared WORDS, so it rates them barely related,")
print("   even though they ask and answer the very same thing.")

And because a bag keeps no record of order, *"the dog bit the man"* and *"the man bit the dog"* — opposite meanings — collapse to the identical vector. Word order, which often carries the meaning, is simply thrown away.

In [ ]:
dog_vocab = build_vocabulary(["the dog bit the man"])
v1 = bag_of_words("the dog bit the man", dog_vocab)
v2 = bag_of_words("the man bit the dog", dog_vocab)
print("'the dog bit the man' bag :", v1.tolist())
print("'the man bit the dog' bag :", v2.tolist())
print("identical bag?", bool(np.array_equal(v1, v2)), " (order is thrown away)")

Both naive methods fail for the same root reason: **they encode the surface form of text, the literal characters, and never the meaning.** What we want is the opposite — a representation where the *meaning* drives the numbers, so that "refund window" and "refunds within 30 days" come out *close*, and "refund window" and "banana bread recipe" come out *far apart*. That representation is an embedding.

## Step 3 — Dense embeddings: the real thing

An **embedding** is a **dense vector** (most or all entries carry a real, nonzero value) that represents text so that *meaning becomes position in space*: similar meanings get vectors that sit close together, different meanings get vectors that sit far apart. Where a one-hot vector wasted 100,000 slots to say one thing, a dense embedding might use a few hundred slots, *each* contributing a little to the meaning.

Nobody hand-assigns these numbers. They're **learned** by an **embedding model** — a neural network trained on enormous text to place similar text close and dissimilar text far, via the **distributional hypothesis**: *"you shall know a word by the company it keeps."*

Here is the intended path — the exact illustrative snippet from the essay. Text in, a fixed-length dense vector out:

```python
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")   # 384 dimensions
vector = model.encode("Refunds are accepted within 30 days of purchase.")
print(len(vector))   # 384  -> this model returns 384 numbers, every time
```

That real library path is the **headline** — the code you'd actually ship. To keep this notebook runnable with no model and no network, we wrap the load in `try/except`. If the import or the cached weights aren't there, we fall back to a transparent stand-in. This is the **canonical guarded-loader pattern** the later parts reuse.

In [ ]:
def load_real_model():
    """Return a real SentenceTransformer encoder, or None if it can't load offline."""
    try:
        from sentence_transformers import SentenceTransformer

        model = SentenceTransformer("all-MiniLM-L6-v2")  # 384 dimensions

        def encode(texts):
            # normalize_embeddings=True -> every vector is unit length, so a dot
            # product equals cosine similarity (the trick from Part 3).
            return np.asarray(model.encode(texts, normalize_embeddings=True))

        return encode
    except Exception as exc:  # missing package, no network, no cached weights...
        print(f"[real model unavailable: {type(exc).__name__}] -> using offline fallback")
        return None

### Step 3b — the transparent offline fallback

A *real* embedding model learned, from billions of sentences, to place similar meanings near each other. We can't reproduce that without the model. What we *can* do, with zero dependencies, is mimic the **interface**: any text → a fixed-length dense unit vector, deterministic and comparable in one space.

How: hash each token into the vector's dimensions and accumulate. Same text → same vector, always. Texts that *share words* land closer (shared tokens push the same dimensions the same way), so it behaves a little like bag-of-words projected into a short dense vector — enough to show the *shape* and the *geometry*, while being honest that it does **not** understand synonyms. We deliberately mirror the real model's 384 dimensions.

In [ ]:
FALLBACK_DIM = 384  # mirror all-MiniLM-L6-v2's 384 dimensions on purpose.


def _hashed_token_vector(token, dim):
    # Deterministic hash -> a reproducible pseudo-random direction for this token.
    h = hashlib.sha256(token.encode("utf-8")).digest()
    # Stretch the 32-byte digest into `dim` floats in [-1, 1], deterministically.
    raw = np.frombuffer((h * (dim // len(h) + 1))[:dim], dtype=np.uint8)
    return (raw.astype(np.float64) / 255.0) * 2.0 - 1.0


def _fallback_encode_one(text, dim=FALLBACK_DIM):
    tokens = tokenize(text)
    if not tokens:
        return np.zeros(dim)
    vec = np.zeros(dim)
    for token in tokens:
        vec += _hashed_token_vector(token, dim)
    vec /= len(tokens)  # average so length doesn't depend on token count
    return unit(vec)    # unit length -> dot product == cosine (Part 3)


def fallback_encode(texts):
    return np.vstack([_fallback_encode_one(t) for t in texts])

Now pick the embedder *once*. We try the real model; if it isn't there, we drop to the fallback and say so clearly. Everything downstream calls the same `encode` either way — that uniform interface is the whole point of the guarded pattern.

In [ ]:
encode = load_real_model()
using_real = encode is not None
if not using_real:
    encode = fallback_encode

label = "REAL all-MiniLM-L6-v2" if using_real else "offline hashing fallback"
print("Embedder in use:", label)

### Text in, numbers out

Let's make "text in, a fixed-length dense vector out" literal. We embed the chunk and the query *into the same space*. The exact numbers are meaningless to a human — what matters is the **shape** (a fixed-length dense vector) and the **promise** (similar meanings produce nearby vectors).

In [ ]:
chunk_vec = encode([CHUNK])[0]
query_vec = encode([QUERY])[0]

print(f"encode(chunk): {len(chunk_vec)} numbers   (the model returns this many, every time)")
print(f"  first 4 values: {np.round(chunk_vec[:4], 3).tolist()}")
print(f"encode(query): {len(query_vec)} numbers   (same space -> directly comparable)")

# The payoff: score the chunk against the query with cosine -- the SAME comparison that
# gave bag-of-words a near-zero -- then contrast with an unrelated banana-bread sentence.
score_query     = cosine(chunk_vec, query_vec)
unrelated       = "Here is a banana bread recipe with ripe bananas."
unrelated_vec   = encode([unrelated])[0]
score_unrelated = cosine(chunk_vec, unrelated_vec)

print(f"cosine(chunk, query)     = {score_query:.3f}")
print(f"cosine(chunk, unrelated) = {score_unrelated:.3f}")
print()
if using_real:
    print("-> a real model places 'refund window?' next to 'refunds within 30 days'")
    print("   because they keep the same company. THAT is search by meaning.")
else:
    print("-> the fallback is only a hash of shared words, so don't read meaning into it;")
    print("   install sentence-transformers to see the genuine 'close in meaning' effect.")

## Step 4 — `king − man + woman ≈ queen`

The classic demonstration: meaning becomes **geometry**, so *"subtract man, add woman"* is a little journey meaning *change the gender, keep the royalty*. Start at `king`, take that step, and you land near `queen`. We compute the analogy vector and rank candidates by closeness to it.

An honest caveat the essay insists on: that clean arithmetic comes from older *per-word* embeddings (word2vec / GloVe). Modern *sentence* embeddings are richer and won't do crisp subtraction, and the hashing fallback can't do it at all (a hash of `king` is unrelated to a hash of `queen`). So the helper below is **real-model-only**; on the fallback we keep the intuition and skip the fake numbers.

In [ ]:
def analogy(encode, a, b, c, candidates):
    """Rank candidates by closeness to (a - b + c). E.g. king - man + woman."""
    va, vb, vc = encode([a]), encode([b]), encode([c])
    target = unit((va - vb + vc)[0])
    ranked = [(word, cosine(target, encode([word])[0])) for word in candidates]
    ranked.sort(key=lambda pair: -pair[1])
    return ranked


candidates = ["queen", "king", "prince", "princess", "banana", "throne", "woman", "man"]

if using_real:
    ranked = analogy(encode, "king", "man", "woman", candidates)
    print("Nearest words to (king - man + woman), best first:")
    for word, score in ranked[:5]:
        print(f"  {word:<10} cosine = {score:+.3f}")
    print("-> 'queen' should rank at or near the top: the DIRECTION captured a concept.")
else:
    print("This party trick needs genuine learned word geometry (word2vec/GloVe-style).")
    print("The offline hashing fallback has no such geometry -- a hash of 'king' is")
    print("unrelated to a hash of 'queen' -- so we skip the numbers rather than fake them.")
    print("Intuition to keep: 'subtract man, add woman' = change gender, keep royalty,")
    print("so the journey from king lands you in queen's neighborhood.")

## Back to RAG: now the **embed** station makes sense

In Part 1 the pipeline had a station called **embed** you took on faith. Now it's concrete:

- **At indexing time** (once, ahead of questions): run every chunk through the embedding model, get a dense vector per chunk, save them in the **vector store**. Your knowledge base is now a cloud of points in meaning-space, one point per chunk.
- **At query time** (per question): run the *user's question* through the *same* model, getting one more point in the same space. "Find the relevant chunks" becomes "**find the stored points closest to the question's point.**" Closest in space is closest in meaning, which is most relevant.

Below is a miniature of exactly that loop: a tiny corpus of chunks, embedded once, then a query embedded and scored against each. The top hit is what retrieval would hand the model.

In [ ]:
knowledge_base = [
    "Refunds are accepted within 30 days of purchase.",
    "Our office is open Monday to Friday, 9am to 5pm.",
    "Here is a banana bread recipe with ripe bananas.",
    "Shipping is free on orders over fifty dollars.",
]

# Index time: embed every chunk once.
kb_vectors = encode(knowledge_base)

# Query time: embed the question, score it against each stored chunk.
q_vec = encode([QUERY])[0]
scored = [(cosine(q_vec, kb_vectors[i]), chunk) for i, chunk in enumerate(knowledge_base)]
scored.sort(key=lambda pair: -pair[0])

print(f"query: {QUERY}\n")
for score, chunk in scored:
    print(f"  {score:+.3f}  {chunk}")
print(f"\nTop hit -> {scored[0][1]!r}")

Even on the offline fallback the refund chunk wins here, because it literally shares the word *refund* with the query. With the real model it wins for the deeper reason — the question and the chunk *keep the same company* in meaning-space — and it would still win for paraphrases that share no words at all. **That** is the capability the rest of the pipeline is built on.

## What you learned

- Computers compare **numbers**, not **meanings**, so search by meaning requires first turning text into numbers in a way that preserves meaning.
- The naive encodings fail instructively: **one-hot** and **bag-of-words** produce huge, mostly empty **sparse** vectors that capture spelling, not sense — so `king` is no closer to `queen` than to `banana`, and order is thrown away.
- An **embedding** is a compact **dense vector** that represents text so that *meaning becomes position*: similar meanings land close, different meanings land far.
- An **embedding model** learns this layout from raw text via the **distributional hypothesis**. RAG uses **sentence / passage** embeddings, not single-word ones.
- The `king − man + woman ≈ queen` arithmetic shows directions in the space carry meaning — a property of older per-word embeddings, kept here behind a real-model-only guard.
- This is what makes RAG's **retrieve** step work: embed the chunks, embed the question into the same space, and "relevant" simply means "closest point."

**The offline guard you just used** — `load_real_model()` returning `None` so a transparent, deterministic stand-in (`fallback_encode`) keeps every cell runnable — is the canonical pattern reused by every later part. The real library is always the headline; the stand-in just keeps the lights on with no network and no API key.

## Next

We've leaned hard on the words *close* and *closest*. **Part 3 — Measuring Similarity** makes them precise: how do you actually measure the distance between two vectors, why is **cosine similarity** the usual answer for embeddings, and what changes when you do it across millions of points?

📖 https://www.mefby.com/essays/measuring-similarity